# Projeto (Seção 9) — solução do instrutor

## Especificação

Campo calculado e cacheado `registration_country_code` (serializado como `registrationCountryCode`) a partir de `country_code_lookup`;
`repr` só com `id_`, `manufacturer`, `series_name` e `type_`. Enunciado em [`04_projeto`](04_projeto.ipynb).

In [1]:
from datetime import date
from enum import Enum
from typing import Annotated, TypeVar
from uuid import UUID, uuid4

from pydantic import (
    AfterValidator,
    BaseModel,
    ConfigDict,
    Field,
    field_serializer,
    field_validator,
    UUID4,
    ValidationInfo,
)
from pydantic.alias_generators import to_camel


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


BoundedString = Annotated[str, Field(min_length=2, max_length=50)]

T = TypeVar('T')

BoundedList = Annotated[list[T], Field(min_length=1, max_length=5)]

countries = {
    "australia": ("Australia", "AUS"),
    "canada": ("Canada", "CAN"),
    "china": ("China", "CHN"),
    "france": ("France", "FRA"),
    "germany": ("Germany", "DEU"),
    "india": ("India", "IND"),
    "mexico": ("Mexico", "MEX"),
    "norway": ("Norway", "NOR"),
    "pakistan": ("Pakistan", "PAK"),
    "san marino": ("San Marino", "SMR"),
    "sanmarino": ("San Marino", "SMR"),
    "spain": ("Spain", "ESP"),
    "sweden": ("Sweden", "SWE"),
    "united kingdom": ("United Kingdom", "GBR"),
    "uk": ("United Kingdom", "GBR"),
    "great britain": ("United Kingdom", "GBR"),
    "britain": ("United Kingdom", "GBR"),
    "us": ("United States of America", "USA"),
    "united states": ("United States of America", "USA"),
    "usa": ("United States of America", "USA"),
}

valid_country_names = sorted(countries.keys())

def lookup_country(name: str) -> tuple[str, str]:
    name = name.strip().casefold()

    try:
        return countries[name]
    except KeyError:
        raise ValueError(
            "Unknown country name. "
            f"Country name must be one of: {','.join(valid_country_names)}"
        )

Country = Annotated[str, AfterValidator(lambda name: lookup_country(name)[0])]

In [2]:
country_code_lookup = {
    name: code
    for name, code in countries.values()
}
print(f"country_code_lookup: {country_code_lookup!r}")

country_code_lookup: {'Australia': 'AUS', 'Canada': 'CAN', 'China': 'CHN', 'France': 'FRA', 'Germany': 'DEU', 'India': 'IND', 'Mexico': 'MEX', 'Norway': 'NOR', 'Pakistan': 'PAK', 'San Marino': 'SMR', 'Spain': 'ESP', 'Sweden': 'SWE', 'United Kingdom': 'GBR', 'United States of America': 'USA'}


In [3]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4 Competition xDrive",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "topFeatures": ["6 cylinders", "all-wheel drive", "convertible"],
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "us",
    "registrationDate": "2023-06-01",
    "licensePlate": "AAA-BBB"
}

expected_serialized_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4 Competition xDrive',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'topFeatures': ['6 cylinders', 'all-wheel drive', 'convertible'],
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'United States of America',
    'registrationCountryCode': 'USA',
    'registrationDate': date(2023, 6, 1),
    'licensePlate': 'AAA-BBB',
}

## Solução

Para implementar, usamos `@cached_property` com o decorador `@computed_field` do Pydantic.

In [4]:
from functools import cached_property
from pydantic import computed_field

In [5]:
class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=to_camel,
    )

    id_: UUID4 | None = Field(alias="id", default_factory=uuid4)
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType = Field(alias="type")
    is_electric: bool = Field(default=False, repr=False)
    manufactured_date: date = Field(
        validation_alias="completionDate",
        ge=date(1980, 1, 1),
        repr=False
    )
    base_msrp_usd: float = Field(
        validation_alias="msrpUSD",
        serialization_alias="baseMSRPUSD",
        repr=False,
    )
    top_features: BoundedList[BoundedString] | None = Field(default=None, repr=False)
    vin: BoundedString = Field(repr=False)
    number_of_doors: int = Field(
        default=4,
        validation_alias="doors",
        ge=2,
        le=4,
        multiple_of=2,
        repr=False,
    )
    registration_country: Country | None = Field(default=None, repr=False)

    @computed_field(repr=False)
    @cached_property
    def registration_country_code(self) -> str:
        return country_code_lookup[self.registration_country]

    registration_date: date | None = Field(default=None, repr=False)
    license_plate: BoundedString | None = Field(default=None, repr=False)


    @field_serializer("manufactured_date", "registration_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")

    @field_validator("registration_date")
    @classmethod
    def validate_registration_date(cls, value:date, values: ValidationInfo):
        data = values.data
        if "manufactured_date" in data and data["manufactured_date"] > value:
            raise ValueError("Automobile cannot be registered prior to manufacture date.")
        return value

Desserializando um modelo e conferindo que a representação foi limpa e que o código do país aparece na serialização:

In [6]:
car = Automobile.model_validate(data)
print(f"car: {car!r}")

car: Automobile(id_=UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'), manufacturer='BMW', series_name='M4 Competition xDrive', type_=<AutomobileType.convertible: 'Convertible'>)


In [7]:
assert car.model_dump(by_alias=True) == expected_serialized_by_alias

> 📝 **Nota de revisão — dois bugs nesta solução:**
>
> 1. **País ausente quebra a serialização.** `country_code_lookup[self.registration_country]` com `registration_country=None` levanta
>    `KeyError` — e o campo calculado é avaliado em todo `model_dump()`. (Além disso, o tipo de retorno deveria ser `str | None`.)
> 2. **Cache desatualizado.** O campo é cacheado, mas `registration_country` continua mutável (`validate_assignment=True`, sem `frozen`).
>    Depois do primeiro acesso, trocar o país não atualiza o código. A própria aula de propriedades avisou: cachear exige congelar a origem.
>
> Também herda o `TypeError` da seção 8 (carro sem `registrationDate`). Provas:

In [8]:
car_no_country = {k: v for k, v in data.items() if k != "registrationCountry"}
try:
    Automobile.model_validate(car_no_country).model_dump()
except Exception as ex:
    print(f"1) sem país: {type(ex).__name__}: {ex}")

car = Automobile.model_validate(data)
car.registration_country_code                       # primeiro acesso: cacheia 'USA'
car.registration_country = "france"
print(f"2) país trocado para {car.registration_country}, código continua {car.registration_country_code}")

1) sem país: KeyError: None
2) país trocado para France, código continua USA
